# E02 — seed ensembles of E03 (validation only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Then Runtime → Run all; about 30–35 minutes. Keep the tab open.

**Design fixed before any run (experiment ledger, E02):** 9 seeds of the exact E03 configuration, in three fixed groups:
A = 20261015/16/17 (already trained in E03), B = 20261018/19/20, C = 20261021/22/23. This notebook trains the 6 new seeds.
Each group's class probabilities are averaged and the most likely class is called → 3 ensemble scores (mean ± std).
The same 9 runs scored one seed at a time are reported alongside, so the ensemble is compared with single models.

The training script must be byte-identical to the E03 run (sha256 `5e9d990c…`); the notebook stops otherwise.
Outputs in `MyDrive/Senior/results/`: `main_model_e03_s2.json` + `_preds.csv.gz`, `ensemble_e02.json`.


In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, os
assert os.path.exists('scripts/seed_ensemble.py'), 'stop: push scripts/seed_ensemble.py from your PC, then run again'
sha = hashlib.sha256(open('scripts/baselines_deep.py', 'rb').read()).hexdigest()
print('baselines_deep.py', sha[:12])
assert sha.startswith('5e9d990c'), 'stop: baselines_deep.py differs from the E03 run, so the new seeds would not be like for like'
assert os.path.exists(f'{OUT_DIR}/main_model_e03_preds.csv.gz'), 'stop: E03 predictions (seeds 15-17) not found in Drive results'


In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo "$OUT_DIR"
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
# Files must match the PC copies (hashes recorded 2026-10-15; sentiment file as stored in git, LF line endings)
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Pipeline and look-ahead tests; the notebook stops here if either fails
import subprocess
for t in ['tests/test_pipeline.py', 'tests/test_no_lookahead.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed'


In [ ]:
# 6 new seeds, E03 configuration unchanged (about 25-30 minutes)
!python scripts/baselines_deep.py --models multiscale_sector --news data/derived/sentiment_scores.csv --extra-features e03 \
    --lr 1e-4 --dropout 0.2 --max-epochs 30 --patience 10 \
    --seeds 20261018 20261019 20261020 20261021 20261022 20261023 \
    --out "$OUT_DIR/main_model_e03_s2.json"


In [ ]:
# E02: three seed ensembles of three, against the same 9 seeds scored one at a time
!python scripts/seed_ensemble.py --preds "$OUT_DIR/main_model_e03_preds.csv.gz" "$OUT_DIR/main_model_e03_s2_preds.csv.gz" \
    --model multiscale_sector --groups 20261015,20261016,20261017 20261018,20261019,20261020 20261021,20261022,20261023 \
    --out "$OUT_DIR/ensemble_e02.json"
